<a href="https://colab.research.google.com/github/alicsrsustain-sudo/HVAC-Optimization-/blob/main/AHU_Motor_Speed_Reduction_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Motor Speed Reduction Calculator
Uses the Affinity Law: Power ∝ Speed³
Reducing a motor's speed via a VFD cuts power consumption by the cube of the speed ratio.

Add, remove, or edit motors in the MOTORS list. Change cost_per_kwh to match your tariff.
"""

# ─────────────────────────────────────────────
#  INPUTS — change any value and re-run
# ─────────────────────────────────────────────

cost_per_kwh = 0.28   # € per kWh

# Each motor is a dict with:
#   name            - label for the output table
#   motor_size_kw   - rated motor size in kW
#   hours_per_year  - annual running hours
#   efficiency      - motor efficiency as a decimal (e.g. 0.9 = 90%)
#   current_speed   - current speed as a fraction of full speed (1.0 = 100%)
#   new_speed       - proposed speed as a fraction of full speed (e.g. 0.65 = 65%)

motors = [
    {"name": "AHU1 - Motor 1", "motor_size_kw": 4, "hours_per_year": 3120, "efficiency": 0.90, "current_speed": 1.00, "new_speed": 0.50},
    {"name": "AHU1 - Motor 2", "motor_size_kw": 4, "hours_per_year": 3120, "efficiency": 0.90, "current_speed": 1.00, "new_speed": 0.65},
    {"name": "AHU2 - Motor 1", "motor_size_kw": 4, "hours_per_year": 3120, "efficiency": 0.90, "current_speed": 1.00, "new_speed": 0.65},
    {"name": "AHU2 - Motor 2", "motor_size_kw": 4, "hours_per_year": 3120, "efficiency": 0.90, "current_speed": 1.00, "new_speed": 0.65},
]


# ─────────────────────────────────────────────
#  CALCULATIONS
# ─────────────────────────────────────────────

def calculate_motor(motor, cost_per_kwh):
    """
    Affinity Law: shaft power scales with the cube of speed ratio.
    Electrical input power = shaft power / efficiency.
    """
    size       = motor["motor_size_kw"]
    hours      = motor["hours_per_year"]
    eff        = motor["efficiency"]
    spd_cur    = motor["current_speed"]
    spd_new    = motor["new_speed"]

    # Electrical input power at each speed
    input_power_current  = (size * spd_cur**3) / eff   # kW
    input_power_new      = (size * spd_new**3) / eff   # kW

    # Annual energy & cost
    current_cost = input_power_current * hours * cost_per_kwh   # €
    new_cost     = input_power_new     * hours * cost_per_kwh   # €
    saving       = current_cost - new_cost                       # €

    return {
        "Current Power (kW)" : input_power_current,
        "New Power (kW)"     : input_power_new,
        "Current Cost (€)"   : current_cost,
        "New Cost (€)"       : new_cost,
        "Saving (€)"         : saving,
    }

results = [calculate_motor(m, cost_per_kwh) for m in motors]


# ─────────────────────────────────────────────
#  OUTPUT
# ─────────────────────────────────────────────

col_name = 18
col_num  = 8

# Header
print("=" * 95)
print("  MOTOR SPEED REDUCTION CALCULATOR")
print("=" * 95)
print(f"  Cost per kWh: €{cost_per_kwh:.4f}")

# Table header
print("\n" + "-" * 95)
print(
    f"  {'Motor':<{col_name}} "
    f"{'kW':>{col_num}} "
    f"{'Hrs/yr':>{col_num}} "
    f"{'Effic.':>{col_num}} "
    f"{'Cur Spd':>{col_num}} "
    f"{'New Spd':>{col_num}} "
    f"{'Cur Cost €':>{col_num+2}} "
    f"{'New Cost €':>{col_num+2}} "
    f"{'Saving €':>{col_num+2}}"
)
print("-" * 95)

total_current = 0
total_new     = 0
total_saving  = 0

for m, r in zip(motors, results):
    print(
        f"  {m['name']:<{col_name}} "
        f"{m['motor_size_kw']:>{col_num}.0f} "
        f"{m['hours_per_year']:>{col_num}.0f} "
        f"{m['efficiency']:>{col_num}.0%} "
        f"{m['current_speed']:>{col_num}.0%} "
        f"{m['new_speed']:>{col_num}.0%} "
        f"{r['Current Cost (€)']:>{col_num+2},.2f} "
        f"{r['New Cost (€)']:>{col_num+2},.2f} "
        f"{r['Saving (€)']:>{col_num+2},.2f}"
    )
    total_current += r["Current Cost (€)"]
    total_new     += r["New Cost (€)"]
    total_saving  += r["Saving (€)"]

# Totals row
print("-" * 95)
print(
    f"  {'TOTAL':<{col_name}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{total_current:>{col_num+2},.2f} "
    f"{total_new:>{col_num+2},.2f} "
    f"{total_saving:>{col_num+2},.2f}"
)
print("=" * 95)

pct_saving = (total_saving / total_current) * 100
print(f"\n  Overall Cost Saving: €{total_saving:,.2f}/year  ({pct_saving:.1f}%)")
print("=" * 95)


  MOTOR SPEED REDUCTION CALCULATOR
  Cost per kWh: €0.2800

-----------------------------------------------------------------------------------------------
  Motor                    kW   Hrs/yr   Effic.  Cur Spd  New Spd Cur Cost € New Cost €   Saving €
-----------------------------------------------------------------------------------------------
  AHU1 - Motor 1            4     3120      90%     100%      50%   3,882.67     485.33   3,397.33
  AHU1 - Motor 2            4     3120      90%     100%      65%   3,882.67   1,066.28   2,816.39
  AHU2 - Motor 1            4     3120      90%     100%      65%   3,882.67   1,066.28   2,816.39
  AHU2 - Motor 2            4     3120      90%     100%      65%   3,882.67   1,066.28   2,816.39
-----------------------------------------------------------------------------------------------
  TOTAL                                                            15,530.67   3,684.17  11,846.50

  Overall Cost Saving: €11,846.50/year  (76.3%)
